In [141]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

In [142]:
{'market': [f'M{i}' for i in range(1, 19)], 
 'economic': [f'E{i}' for i in range(1, 21)],
 'interest': [f'I{i}' for i in range(1, 10)], 
 'price': ['P8', 'P9', 'P10', 'P12', 'P13'],
 'volatility': [f'V{i}' for i in range(1, 14)],
 'sentiment': [f'S{i}' for i in range(1, 13)],
 'dummy': [f'D{i}' for i in range(1, 10)]}

{'market': ['M1',
  'M2',
  'M3',
  'M4',
  'M5',
  'M6',
  'M7',
  'M8',
  'M9',
  'M10',
  'M11',
  'M12',
  'M13',
  'M14',
  'M15',
  'M16',
  'M17',
  'M18'],
 'economic': ['E1',
  'E2',
  'E3',
  'E4',
  'E5',
  'E6',
  'E7',
  'E8',
  'E9',
  'E10',
  'E11',
  'E12',
  'E13',
  'E14',
  'E15',
  'E16',
  'E17',
  'E18',
  'E19',
  'E20'],
 'interest': ['I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9'],
 'price': ['P8', 'P9', 'P10', 'P12', 'P13'],
 'volatility': ['V1',
  'V2',
  'V3',
  'V4',
  'V5',
  'V6',
  'V7',
  'V8',
  'V9',
  'V10',
  'V11',
  'V12',
  'V13'],
 'sentiment': ['S1',
  'S2',
  'S3',
  'S4',
  'S5',
  'S6',
  'S7',
  'S8',
  'S9',
  'S10',
  'S11',
  'S12'],
 'dummy': ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9']}

In [143]:
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

def preprocessing(data, typ):
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13"]
    v1, v2 = 'I', 'S'
    its, data_2 = 10, data.copy()
    
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        for i in range(1, its):
            data[f'{v1}_{v2}_{i}'] = data_2[f'{v1}{i}'] * data_2[f'{v2}{i}']
        
        data = data.rename(columns={'forward_returns': 'target'})
    else: 
        data = data[main_features].copy()
        for i in range(1, its):
            data[f'{v1}_{v2}_{i}'] = data_2[f'{v1}{i}'] * data_2[f'{v2}{i}']
    
    for col in data.columns:
        if col != 'target':
            data[col].fillna(data[col].mean(), inplace=True)
    
    if 'target' in data.columns:
        data = data.dropna(subset=['target'])
    
    return data

train_processed = preprocessing(train, "train")

train_split, val_split = train_test_split(
    train_processed, test_size=0.1, random_state=42
)

train_x = train_split.drop(columns=["target"])
test_x = val_split.drop(columns=["target"])
train_y = train_split['target']
test_y = val_split['target']

In [144]:
train_processed

,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,...,S10,S11,S12,I2,P8,P9,P10,P12,P13,target
0,1.564574,0.504941,0.125869,0.118739,0.012314,0.007005,0.485070,-0.047127,-0.007330,0.097865,...,0.437665,0.432784,0.264418,-0.528014,1.542002,0.395020,1.466365,-0.024978,0.508856,-0.002421
1,1.564574,0.504941,0.125869,0.118739,0.012314,0.007005,0.485070,-0.047127,-0.007330,0.097865,...,0.437665,0.432784,0.264418,-0.528014,1.542002,0.395020,1.466365,-0.024978,0.508856,-0.008495
2,1.564574,0.504941,0.125869,0.118739,0.012314,0.007005,0.485070,-0.047127,-0.007330,0.097865,...,0.437665,0.432784,0.264418,-0.528014,1.542002,0.395020,1.466365,-0.024978,0.508856,-0.009624
3,1.564574,0.504941,0.125869,0.118739,0.012314,0.007005,0.485070,-0.047127,-0.007330,0.097865,...,0.437665,0.432784,0.264418,-0.528014,1.542002,0.395020,1.466365,-0.024978,0.508856,0.004662
4,1.564574,0.504941,0.125869,0.118739,0.012314,0.007005,0.485070,-0.047127,-0.007330,0.097865,...,0.437665,0.432784,0.264418,-0.528014,1.542002,0.395020,1.466365,-0.024978,0.508856,-0.011686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,1.565379,0.184524,0.019180,0.019180,0.005952,0.005952,0.911376,-0.083496,-0.572447,0.223638,...,0.273148,0.134921,0.634465,0.984115,1.784929,0.039683,2.087888,0.648069,0.625331,0.002457
8986,1.562946,0.184193,0.018849,0.018849,0.005622,0.005622,0.911706,-0.083542,-0.572080,0.222910,...,0.933201,0.721561,1.211345,0.904453,1.791596,0.037037,2.092041,0.916799,0.739418,0.002312
8987,1.560520,0.183862,0.018519,0.018519,0.005291,0.005291,0.912037,-0.083874,-0.572016,0.222211,...,0.793651,0.689815,0.885178,0.842295,1.792816,0.041005,2.092283,-0.702456,0.809193,0.002891
8988,1.558102,0.183532,0.018188,0.018188,0.004960,0.004960,0.912368,-0.084206,-0.571952,0.221513,...,0.011905,0.026455,-0.001785,0.858582,1.792934,0.046958,2.094798,1.456942,0.923611,0.008310


In [145]:
LGBM_R_parm = {'boosting_type': 'gbdt', 
               'colsample_bytree': 0.9484106149593443, 
               'learning_rate': 0.1988123373955639, 
               'max_bin': 77, 
               'max_depth': 10, 
               'metric': 'mape', 
               'min_child_samples': 81, 
               'min_data_in_leaf': 21, 
               'n_estimators': 5029, 
               'num_leaves': 42, 
               'objective': 'regression_l1', 
               'reg_alpha': 0.6355835028602363, 
               'reg_lambda': 3.109823217156622, 
               'subsample': 0.7300733288106989, 
               'verbosity': -1}

print(f'LGBMRegressor TRAINING...')
LGBM_R = LGBMRegressor(**LGBM_R_parm)
LGBM_R.fit(train_x, train_y, eval_set=[(test_x, test_y)])
joblib.dump(LGBM_R, 'LGBM_R.pkl')

LGBM_R_model = joblib.load('LGBM_R.pkl')

LGBMRegressor TRAINING...


In [146]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)
MIN_INVESTMENT = 0
MAX_INVESTMENT = 2

class ParticipantVisibleError(Exception):
    pass

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = None) -> float:
    if not pd.api.types.is_numeric_dtype(submission['prediction']):
        raise ParticipantVisibleError('Predictions must be numeric')

    solution = solution.copy()
    solution['position'] = submission['prediction'].values

    if solution['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
    if solution['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].min()} below minimum of {MIN_INVESTMENT}')

    solution['strategy_returns'] = solution['risk_free_rate'] * (1 - solution['position']) + solution['position'] * solution['forward_returns']

    strategy_excess_returns = solution['strategy_returns'] - solution['risk_free_rate']
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solution)) - 1
    strategy_std = solution['strategy_returns'].std()

    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ParticipantVisibleError('Division by zero, strategy std is zero')
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)

    market_excess_returns = solution['forward_returns'] - solution['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solution)) - 1
    market_std = solution['forward_returns'].std()

    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)

    if market_volatility == 0:
        raise ParticipantVisibleError('Division by zero, market std is zero')

    excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
    vol_penalty = 1 + excess_vol

    return_gap = max(
        0,
        (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr,
    )
    return_penalty = 1 + (return_gap**2) / 100

    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)


print("\nEvaluating on validation set...")
val_predictions = LGBM_R_model.predict(test_x)
val_signals = np.array([convert_ret_to_signal(pred) for pred in val_predictions])

val_indices = val_split.index
val_original = train.loc[val_indices].copy()

solution_df = pd.DataFrame({
    'forward_returns': val_original['forward_returns'].values,
    'risk_free_rate': val_original['risk_free_rate'].values
})

submission_df = pd.DataFrame({
    'prediction': val_signals
})

try:
    validation_score = score(solution_df, submission_df)
    print(f"Validation Adjusted Sharpe Ratio: {validation_score}")
except Exception as e:
    print(f"Error calculating validation score: {e}")


Evaluating on validation set...
Validation Adjusted Sharpe Ratio: 0.22868827933658392


In [147]:
def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        
        test_processed = preprocessing(test_df, 'inference')
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))